In [114]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from sklearn.metrics import r2_score, mean_absolute_error

# Predicting Freight Cost

**Objective:** Develop a machine learning model to predict freight costs for vendor invoices based on shipment quantity and invoice value, enabling more accurate cost forecasting and procurement planning.

### Business Impact
- Freight cost is a significant component of total landed cost.
- Inaccurate freight estimates can negatively impact profit margins and inventory planning.
- Automated freight prediction allows procurement teams to estimate total costs before invoices are received.
- Improved forecasting supports budgeting, supplier evaluation, and vendor negotiation strategies.

In [115]:
vender_df = pd.read_csv(r'../data\vendor_invoice.csv')
vender_df.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,NaN
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,NaN


In [116]:
vender_df.shape

(5543, 10)

In [117]:
vender_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5543 entries, 0 to 5542
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   VendorNumber  5543 non-null   int64  
 1   VendorName    5543 non-null   str    
 2   InvoiceDate   5543 non-null   str    
 3   PONumber      5543 non-null   int64  
 4   PODate        5543 non-null   str    
 5   PayDate       5543 non-null   str    
 6   Quantity      5543 non-null   int64  
 7   Dollars       5543 non-null   float64
 8   Freight       5543 non-null   float64
 9   Approval      374 non-null    str    
dtypes: float64(2), int64(3), str(5)
memory usage: 433.2 KB


In [118]:
num_col = vender_df.select_dtypes(include=['number']).columns.tolist()
cat_col = vender_df.select_dtypes(include=['object']).columns.tolist()

C:\Users\Mayank saini\AppData\Local\Temp\ipykernel_23232\2857081770.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_col = vender_df.select_dtypes(include=['object']).columns.tolist()


In [119]:
vender_df['Approval'].value_counts()

Approval
Frank Delahunt    374
Name: count, dtype: int64

In [120]:
vender_df[['Quantity', 'Dollars', 'Freight']].corr()

,Quantity,Dollars,Freight
Quantity,1.000000,0.963831,0.946550
Dollars,0.963831,1.000000,0.985141
Freight,0.946550,0.985141,1.000000


In [121]:
low_qty = vender_df['Quantity'].quantile(0.25)
high_qty = vender_df['Quantity'].quantile(0.75)

In [122]:
vender_df['Freight_per_unit'] = vender_df['Freight']/vender_df['Quantity']

In [123]:
vender_df.loc[vender_df['Quantity'] < low_qty, 'Freight_per_unit'].mean()

np.float64(0.09489854253138316)

In [124]:
vender_df.loc[vender_df['Quantity'] > high_qty, 'Freight_per_unit'].mean()

np.float64(0.049077654690759046)

In [125]:
vender_df['InvoiceDate'] = pd.to_datetime(vender_df['InvoiceDate'])
vender_df['PODate']      = pd.to_datetime(vender_df['PODate'])
vender_df['PayDate']     = pd.to_datetime(vender_df['PayDate'])

vender_df['invoice_to_pay']  = (vender_df['PayDate']    - vender_df['InvoiceDate']).dt.days  
vender_df['po_to_invoice']   = (vender_df['InvoiceDate'] - vender_df['PODate']).dt.days      
vender_df['po_to_pay']       = (vender_df['PayDate']    - vender_df['PODate']).dt.days       

not taking quantity as a feature becasue of high corr with dolla nd keeping doll becuahse having high corr with freight

In [126]:
drop_cols = ['InvoiceDate', 'PODate', 'PayDate', 
             'VendorName', 'Approval',    
             'VendorNumber', 'PONumber',  
             'Quantity',                  
             'Freight_per_unit']          

X = vender_df.drop(columns=drop_cols + ['Freight'])
y = vender_df['Freight']
y = np.log1p(y)

print(X.columns.tolist())
print(X.dtypes)   

['Dollars', 'invoice_to_pay', 'po_to_invoice', 'po_to_pay']
Dollars           float64
invoice_to_pay      int64
po_to_invoice       int64
po_to_pay           int64
dtype: object


In [127]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)


models = {
    'Ridge'           : Ridge(),
    'Lasso'           : Lasso(),
    'ElasticNet'      : ElasticNet(),
    'DecisionTree'    : DecisionTreeRegressor(random_state=42),
    'RandomForest'    : RandomForestRegressor(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'AdaBoost'        : AdaBoostRegressor(n_estimators=100, random_state=42),
    'XGBoost'         : XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'KNN'             : KNeighborsRegressor(n_neighbors=5),
    'SVR'             : SVR(kernel='rbf'),
}

In [128]:
result = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)
    result.append({
        'Model':name,
        'R2': round(r2_score(y_test,y_hat), 4),
        'MAE': round(mean_absolute_error(y_test,y_hat), 4)
    })
result_df = pd.DataFrame(result).sort_values('R2', ascending=False).reset_index(drop=True)

In [129]:
result_df

,Model,R2,MAE
0,GradientBoosting,0.9897,0.0859
1,KNN,0.9884,0.0963
2,XGBoost,0.9868,0.1043
3,RandomForest,0.9868,0.0982
4,DecisionTree,0.9789,0.1179
5,AdaBoost,0.9698,0.3103
6,SVR,0.8999,0.5113
7,Lasso,0.4234,1.4430
8,ElasticNet,0.4234,1.4430
9,Ridge,0.4228,1.4440


In [130]:
model = GradientBoostingRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Train R²:", r2_score(y_train, model.predict(X_train)))
print("Test  R²:", r2_score(y_test,  model.predict(X_test)))

Train R²: 0.9923397274730069
Test  R²: 0.9897359110600756


In [131]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error


param_grid = {
    'n_estimators'  : [100, 200, 300],
    'learning_rate' : [0.05, 0.1, 0.2],
    'max_depth'     : [3, 4, 5],
    'subsample'     : [0.8, 1.0],
}

grid = GridSearchCV(
    estimator  = GradientBoostingRegressor(random_state=42),
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'r2',
    verbose    = 1,
    n_jobs     = -1
)

grid.fit(X_train, y_train)

print("Best Params :", grid.best_params_)
print("Best CV R²  :", round(grid.best_score_, 4))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Params : {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best CV R²  : 0.9902


In [132]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Test  R²  :", round(r2_score(y_test, y_pred), 4))
print("Train R²  :", round(r2_score(y_train, best_model.predict(X_train)), 4))
print("MAE       :", round(mean_absolute_error(y_test, y_pred), 4))

Test  R²  : 0.9897
Train R²  : 0.9915
MAE       : 0.0874


In [133]:
print(y.describe())

count    5543.000000
mean        3.625769
std         2.189211
min         0.019803
25%         1.795087
50%         3.247658
75%         5.440945
max         9.044194
Name: Freight, dtype: float64


In [135]:
import joblib

joblib.dump(best_model, '../models/freight_predictor_gb.pkl')



['../models/freight_predictor_gb.pkl']